# Generate Diverse Synthetic Data with Noise.

In [1]:
import os
import random
import re
import string
import math
from typing import List, Tuple, Dict
import pandas as pd
import numpy as np
from faker import Faker

In [2]:
fake = Faker()

### Config

- This code sets a fixed random seed (42) for both NumPy (np.random) and Python’s built-in random module to ensure reproducibility of random operations. By initializing the random number generators with the same seed value, any random numbers, shuffled data, or randomized processes generated during the program will produce identical results each time the code runs, which is crucial for consistent testing, debugging, and experiments.

In [3]:
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

In [4]:
OUT_CSV = "Synthetic_Transactions_Diverse - Demo.csv"

### Category list (expanded)

In [5]:
CATEGORIES = [
    "Stationery", "Electronics", "Groceries", "Travel", "Food",
    "Clothing", "Healthcare", "Utilities", "Office Supplies",
    "Maintenance", "Training", "Software", "Subscriptions",
    "Entertainment", "Charity", "Fuel", "Shipping", "Furniture",
    "Insurance", "Professional Services", "Legal"
]

### Building blocks for generation

In [6]:
ITEMS = {
    "Stationery": ["notebooks", "pencils", "markers", "files", "folders", "sticky notes", "highlighters"],
    "Electronics": ["laptop", "headphones", "mouse", "keyboard", "tablet", "monitor", "charger"],
    "Groceries": ["milk", "bread", "eggs", "cheese", "apples", "chicken", "lettuce"],
    "Travel": ["plane ticket", "train ticket", "bus ticket", "taxi ride", "hotel stay", "car rental"],
    "Food": ["burger", "pizza", "pasta", "salad", "sushi", "coffee", "sandwich"],
    "Clothing": ["shirt", "jeans", "jacket", "dress", "shoes", "socks"],
    "Healthcare": ["painkillers", "bandages", "vitamins", "antibiotics", "consultation"],
    "Utilities": ["electricity", "internet", "water", "gas"],
    "Office Supplies": ["printer ink", "stapler", "envelopes", "label sheets"],
    "Maintenance": ["repair", "service call", "inspection", "cleaning"],
    "Training": ["course fee", "workshop", "seminar", "certificate"],
    "Software": ["subscription", "license", "SaaS", "tool access"],
    "Subscriptions": ["monthly subscription", "membership", "annual fee"],
    "Entertainment": ["movie tickets", "concert", "event", "streaming subscription"],
    "Charity": ["donation", "charity contribution", "fundraiser"],
    "Fuel": ["diesel", "petrol", "gasoline"],
    "Shipping": ["courier", "shipping fee", "freight"],
    "Furniture": ["desk", "chair", "cabinet", "bookshelf"],
    "Insurance": ["policy premium", "insurance fee"],
    "Professional Services": ["consultant fee", "audit", "financial advisory"],
    "Legal": ["legal fee", "contract review", "retainer"]
}

In [7]:
VERBS = [
    "Bought", "Purchased", "Ordered", "Grabbed", "Picked up", "Paid for", "Spent on", "Acquired", "Procured"
]

In [8]:
EMOJIS = ["", " 😊", " 👍", "💳", "🧾", "📦", "🏷️", "✈️"]

In [9]:
CURRENCIES = ["$", "€", "£", "₹", "CAD$", "AUD$", "JPY¥"]

In [10]:
FREE_TEXT_PHRASES = [
    "for office use", "for the team", "for personal use", "as a gift",
    "urgent purchase", "as part of project X", "for client Y",
    "monthly recurring", "one-off payment", "reimbursement request"
]

In [11]:
CONNECTORS = [
    "and", "as well as", "including", "plus", "along with", "together with"
]

### Helpers: typos, emoji injection, currency formatting

- This function inject_typos takes a text string and randomly introduces small typos into it with a given probability (typo_prob, default 4%) to simulate human typing errors. It loops through each character and, with a small chance, performs one of four random operations: substitution (replace the character with a random lowercase letter), deletion (remove the character), duplication (repeat the character twice), or swap (swap positions with the next character). Non-alphabetic characters are left unchanged. The function then returns the modified string, which resembles text with realistic random typos.

In [12]:
def inject_typos(text: str, typo_prob: float = 0.04) -> str:
    """Randomly replace / transpose / drop characters to simulate typos."""
    # small chance per character to apply a typo operation
    chars = list(text)
    i = 0
    out = []
    while i < len(chars):
        ch = chars[i]
        if ch.isalpha() and random.random() < typo_prob:
            op = random.choice(["sub", "del", "dup", "swap"])
            if op == "sub":
                out.append(random.choice(string.ascii_lowercase))
            elif op == "del":
                # skip this char
                pass
            elif op == "dup":
                out.append(ch)
                out.append(ch)
            elif op == "swap" and i + 1 < len(chars):
                out.append(chars[i+1])
                out.append(ch)
                i += 1
        else:
            out.append(ch)
        i += 1
    return "".join(out)

In [13]:
def insert_emojis(text: str, prob: float = 0.15) -> str:
    if random.random() < prob:
        return text + random.choice(EMOJIS)
    return text

In [14]:
def currency_amount(i: int = None) -> str:
    """Return formatted amount with random currency and formatting variety."""
    if i is None:
        amt = round(random.uniform(2, 1500), 2)
    else:
        # if list of amounts needed, use provided
        amt = round(random.uniform(2, 1500), 2)
    cur = random.choice(CURRENCIES)
    # some variants: space, comma separators, no decimals
    if random.random() < 0.2:
        # integer
        amt_s = f"{int(amt)}"
    else:
        amt_s = f"{amt:,.2f}"
    if cur in ["$","€","£","₹","CAD$","AUD$","JPY¥"]:
        # sometimes put currency after
        if random.random() < 0.15:
            return f"{amt_s} {cur}"
        return f"{cur}{amt_s}"
    return f"{cur}{amt_s}"

In [15]:
def random_date_phrase():
    # produce phrases like "yesterday", "on 2025-10-20", "last Friday"
    if random.random() < 0.6:
        return random.choice(["yesterday", "today", "last week", "on Monday", "on Friday"])
    else:
        return "on " + fake.date_between(start_date='-1y', end_date='today').isoformat()

### Long template generator

In [16]:
LONG_TEMPLATES = [
    # multi-clause purchases, clauses connected by commas/and
    "{verb} {item_list} for {amount_list} at {store}. {date_phrase}, {free_phrase}.",
    "{verb} {item_list} (incl. {extra_item}) totalling {amount_total} via {channel} {emoji} {free_phrase}.",
    "{verb} {item1} and {item2} for {amount1} and {amount2} respectively, then paid {shipping} for delivery to {city}.",
    "Payment of {amount_total} for {item_list} from {store} including taxes and fees, transaction ID {txid}.",
    "Bought {item_list} for {amount_list} at {store} because {free_phrase}. Also purchased {other_item} for {other_amount}.",
    "{verb} {item1} for {amount1} and {item2} for {amount2} at {store}, for project {project_code} - reimbursable."
]


In [17]:
FREE_TEXT_ADDITIONS = [
    "I need this for Monday's meeting.",
    "Used corporate card.",
    "Requesting reimbursement.",
    "Bought during lunch break.",
    "Promotional offer applied.",
    "Supplier gave discount."
]

### Core generator

In [18]:
class SafeDict(dict):
    def __missing__(self, key):
        return "{" + key + "}"

In [19]:
def safe_format(template, **kwargs):
    # Replaces missing placeholders with the literal text {placeholder}
    return template.format_map(SafeDict(**kwargs))

- The build_long_description function generates a realistic, randomized transaction-style text description based on a given category. It first selects a pool of items (either from that category or from all items if the category is missing) and randomly chooses 2–5 of them, assigning each a fake currency amount. Using various pre-defined lists (like LONG_TEMPLATES, VERBS, EMOJIS, etc.), it fills in placeholders with dynamic data such as store names, cities, transaction IDs, dates, amounts, and random details like shipping or tax. The function adds extra realism by occasionally appending free-text phrases, inserting extra currency mentions, adding emojis or punctuation noise, and even injecting typos. The result is a long, natural-looking, slightly messy text that simulates real-world receipts, invoices, or transaction notes.

In [20]:
def build_long_description(category: str) -> str:
    # choose items based on category; if category not in ITEMS pick random
    item_pool = ITEMS.get(category, sum(ITEMS.values(), []))
    # Pick between 2 and 5 items, but not more than what's available
    max_items = len(item_pool)
    if max_items == 0:
        item_pool = ["miscellaneous item"]  # fallback
        max_items = 1
    
    k = min(random.choice([2,3,4,5]), max_items)
    chosen_items = random.sample(item_pool, k=k)
    amounts = [currency_amount() for _ in range(k)]
    amount_list = ", ".join(amounts)
    item_list = ", ".join(chosen_items)
    store = fake.company()
    city = fake.city()
    date_phrase = random_date_phrase()
    free_phrase = random.choice(FREE_TEXT_PHRASES + FREE_TEXT_ADDITIONS)
    verb = random.choice(VERBS)
    emoji = random.choice(EMOJIS)
    channel = random.choice(["online", "in-store", "via mobile app", "on vendor portal", "at checkout"])
    amount_total = currency_amount()
    txid = fake.bothify(text='??-#####-####')
    extra_item = random.choice(item_pool)
    other_item = random.choice(item_pool)
    other_amount = currency_amount()
    project_code = "PRJ-" + str(random.randint(100,999))
    template = random.choice(LONG_TEMPLATES)
    text = safe_format(
        template,
        verb=verb, item_list=item_list, amount_list=amount_list,
        store=store, date_phrase=date_phrase, free_phrase=free_phrase,
        emoji=emoji, channel=channel, amount_total=amount_total,
        txid=txid, extra_item=extra_item, city=city,
        item1=chosen_items[0], item2=chosen_items[1] if len(chosen_items)>1 else chosen_items[0],
        amount1=amounts[0], amount2=amounts[1] if len(amounts)>1 else amounts[0],
        other_item=other_item, other_amount=other_amount, project_code=project_code,
        shipping=f"${random.uniform(3, 25):.2f}",  # added placeholder for safety
        tax=f"${random.uniform(1, 15):.2f}",       # optional realism boost
    )

    # add free-text addition sometimes
    if random.random() < 0.25:
        text += " " + random.choice(FREE_TEXT_ADDITIONS)

    # chance to mix currencies in sentence (multilingual currency example)
    if random.random() < 0.15:
        # insert another currency mention
        text += f" Also paid {currency_amount()} separately."

    # occasionally add emojis or special chars
    text = insert_emojis(text, prob=0.6)

    # randomly add noise / extra punctuation
    if random.random() < 0.12:
        text = text.replace(".", ". " + random.choice(["Note:", "FYI:", "Reminder:"]))

    # add typos with some probability
    if random.random() < 0.3:
        text = inject_typos(text, typo_prob=0.03)

    return text

In [21]:
def generate_dataset(n: int = 20000, categories: List[str] = None) -> pd.DataFrame:
    if categories is None:
        categories = CATEGORIES
    rows = []
    for _ in range(n):
        cat = random.choice(categories)
        text = build_long_description(cat)
        rows.append({"text_description": text, "category": cat})
    df = pd.DataFrame(rows)
    return df

In [22]:
if __name__ == "__main__":
    print("Generating dataset...")
    df = generate_dataset(n=15000)   # customize size
    df.to_csv(OUT_CSV, index=False)
    print("Saved dataset to", OUT_CSV)

Generating dataset...
Saved dataset to Synthetic_Transactions_Diverse - Demo.csv


In [23]:
df.head()

,text_description,category
0,Procured legal fee and contract review for €36...,Legal
1,"Bought annual fee, membership for CAD$1,244.45...",Subscriptions
2,"Spent onn charger, laptop, mouse for $602.94, ...",Electronics
3,"Procured legal fee, contract review (incl. leg...",Legal
4,"Bought chair and desk for 1,453. Note:13 CAD$ ...",Furniture


In [27]:
df.shape

(15000, 2)